In [15]:
import time
import json
import numpy as np
from collections import Counter
from difflib import get_close_matches

# ============================================================
# LOAD DATA
# ============================================================
with open("clean_documents.json", "r") as f:
    CLEAN_DOCS = json.load(f)

with open("tfidf_index.json", "r") as f:
    TFIDF = json.load(f)

tfidf_matrix = np.array(TFIDF["matrix"])
vocab = TFIDF["vocab"]
vocab_index = {w: i for i, w in enumerate(vocab)}
idf = np.array(TFIDF["idf"])
filenames = TFIDF["filenames"]

# ============================================================
# PREPROCESS & UTILS
# ============================================================
def preprocess_query(q):
    return q.lower().split()

def compute_tf(tokens):
    vec = np.zeros(len(vocab_index))
    counter = Counter(tokens)
    for w, f in counter.items():
        if w in vocab_index:
            vec[vocab_index[w]] = f
    return vec

def cosine_similarity(a, b):
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def autocorrect_safe(tokens):
    corrected = []
    for t in tokens:
        matches = get_close_matches(t, vocab, n=1, cutoff=0.75)
        if matches:
            corrected.append(t)
        else:
            corrected.append(t)
    return corrected, None

def jaccard_similarity(q_tokens, doc_tokens, title_tokens):
    q = set(q_tokens)
    d = set(doc_tokens)

    if len(q) == 0:
        return 0.0

    inter = len(q & d)
    union = len(q | d)
    score = inter / union if union > 0 else 0.0

    if len(q & set(title_tokens)) > 0:
        score += 0.3

    return score

# ============================================================
# SEARCH FUNCTIONS
# ============================================================
def search_tfidf(q_tokens):
    q_tf = compute_tf(q_tokens)
    q_vec = q_tf * idf
    scores = [(i, cosine_similarity(q_vec, doc_vec)) for i, doc_vec in enumerate(tfidf_matrix)]
    scores = [(i, s) for (i, s) in scores if s > 0]
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return [filenames[i] for i, _ in scores]

def search_jaccard(q_tokens):
    scored = []
    for doc in CLEAN_DOCS:
        score = jaccard_similarity(q_tokens, doc["tokens"], doc["title"].lower().split())
        if score > 0:
            scored.append((doc["title"], score))
    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    return [title for title, _ in scored]

# ============================================================
# EVALUATION FUNCTIONS
# ============================================================
def precision_at_k(results, relevant, k):
    results = results[:k]
    hits = sum(1 for r in results if r in relevant)
    return hits / max(len(results), 1)

def recall_at_k(results, relevant, k):
    results = results[:k]
    hits = sum(1 for r in results if r in relevant)
    return hits / len(relevant)

def f1_score(p, r):
    if p + r == 0:
        return 0.0
    return 2 * (p * r) / (p + r)

def evaluate_single(query, relevant_docs, k=10):
    q_tokens = preprocess_query(query)
    q_tokens, _ = autocorrect_safe(q_tokens)

    # TF-IDF
    start = time.time()
    tfidf_results = search_tfidf(q_tokens)
    tfidf_runtime = (time.time() - start) * 1000

    tfidf_precision = precision_at_k(tfidf_results, relevant_docs, k)
    tfidf_recall = recall_at_k(tfidf_results, relevant_docs, k)
    tfidf_f1 = f1_score(tfidf_precision, tfidf_recall)

    # Jaccard
    start = time.time()
    jaccard_results = search_jaccard(q_tokens)
    jaccard_runtime = (time.time() - start) * 1000

    jaccard_precision = precision_at_k(jaccard_results, relevant_docs, k)
    jaccard_recall = recall_at_k(jaccard_results, relevant_docs, k)
    jaccard_f1 = f1_score(jaccard_precision, jaccard_recall)

    return {
        "query": query,
        "tfidf": {
            "runtime": tfidf_runtime,
            "precision": tfidf_precision,
            "recall": tfidf_recall,
            "f1": tfidf_f1
        },
        "jaccard": {
            "runtime": jaccard_runtime,
            "precision": jaccard_precision,
            "recall": jaccard_recall,
            "f1": jaccard_f1
        }
    }

# ============================================================
# TEST SET
# ============================================================
TEST_SET = [
    ("interstellar", ["Interstellar"]),
    ("i am legend", ["I Am Legend"]),
    ("avengers", ["Avengers Endgame", "Avengers Age of Ultron", "The Avengers"]),
    ("spider", [
        "Spider-Man No Way Home", "Spider-Man Homecoming",
        "Spider-Man Far From Home", "Spider-Man Across the Spider-Verse"
    ]),
    ("the batman", ["The Batman", "The Batman Part II"])
]

# ============================================================
# OUTPUT
# ============================================================
def run_evaluation_display():
    for query, relevant in TEST_SET:
        result = evaluate_single(query, relevant)

        print("\n======================================")
        print(f"Evaluasi Query: {query}\n")

        tf = result["tfidf"]
        jc = result["jaccard"]

        print("Cosine Similarity")
        print(f"Runtime   : {tf['runtime']:.4f} ms")
        print(f"Precision : {tf['precision']:.4f}")
        print(f"Recall    : {tf['recall']:.4f}")
        print(f"F1 Score  : {tf['f1']:.4f}\n")

        print("--------------------------------------\n")

        print("Jaccard Similarity")
        print(f"Runtime   : {jc['runtime']:.4f} ms")
        print(f"Precision : {jc['precision']:.4f}")
        print(f"Recall    : {jc['recall']:.4f}")
        print(f"F1 Score  : {jc['f1']:.4f}")

        print("======================================\n")

# ============================================================
# RUN
# ============================================================
run_evaluation_display()



Evaluasi Query: interstellar

Cosine Similarity
Runtime   : 7.5259 ms
Precision : 1.0000
Recall    : 1.0000
F1 Score  : 1.0000

--------------------------------------

Jaccard Similarity
Runtime   : 10.5851 ms
Precision : 1.0000
Recall    : 1.0000
F1 Score  : 1.0000


Evaluasi Query: i am legend

Cosine Similarity
Runtime   : 7.5781 ms
Precision : 0.2500
Recall    : 1.0000
F1 Score  : 0.4000

--------------------------------------

Jaccard Similarity
Runtime   : 7.0002 ms
Precision : 0.1111
Recall    : 1.0000
F1 Score  : 0.2000


Evaluasi Query: avengers

Cosine Similarity
Runtime   : 7.2267 ms
Precision : 0.5000
Recall    : 0.6667
F1 Score  : 0.5714

--------------------------------------

Jaccard Similarity
Runtime   : 7.3202 ms
Precision : 0.5000
Recall    : 0.6667
F1 Score  : 0.5714


Evaluasi Query: spider

Cosine Similarity
Runtime   : 7.5662 ms
Precision : 0.3333
Recall    : 0.7500
F1 Score  : 0.4615

--------------------------------------

Jaccard Similarity
Runtime   : 6.9788